<a href="https://colab.research.google.com/github/sankeawthong/Project-1-Lita-Chatbot/blob/main/%5B20260423%5D%20SecureNet%20LSTM%20Stack%20Sweep%20%E2%80%94%20WSN-BFSF%20(4%20classes).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

"""
SecureNet LSTM Stack Sweep — WSN-BFSF (4 classes)
===================================================
Trains three LSTM stack configurations under identical conditions:
  - 32/16/8  (shallow)
  - 64/32/16 (Pareto-optimal, original)
  - 128/64/32 (deep)

Pipeline mirrors the original notebook exactly:
  SMOTE-NearMiss (fold-contained) → RF (100 trees, top-k features)
  → LSTM stack → Dense+Softmax

Logs per configuration:
  1. Held-out test ACC (%)
  2. Per-class recall + worst-class (minority) recall
  3. Static model file size (MB, .keras SavedModel)
  4. Held-out test FPR (macro and per-class)
  5. Validation ACC curve (final epoch)

Usage:
  python securenet_stack_sweep.py --dataset path/to/wsnbfsf.csv
"""


In [ ]:
pip install imbalanced-learn scikit-learn tensorflow pandas numpy

In [ ]:
import argparse
import os
import sys
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
import tempfile

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────
RANDOM_STATE   = 42
NUM_CLASSES    = 4
NUM_FOLDS      = 10
EPOCHS         = 20
BATCH_SIZE     = 64
TOP_K          = 6          # RF feature selection: top-k (matches paper: 18→6 on WSN-BFSF approx)
L2_REG         = 0.0001
DROPOUT        = 0.2
OUTER_TEST_SIZE = 0.20

# Three stack configurations to evaluate
STACKS = {
    "32/16/8":   [32, 16, 8],
    "64/32/16":  [64, 32, 16],   # original / Pareto-optimal
    "128/64/32": [128, 64, 32],
}

CLASS_NAMES = ["Normal", "Blackhole", "Flooding", "Forwarding"]


# ─────────────────────────────────────────────
# Helper: build LSTM model for a given stack
# ─────────────────────────────────────────────
def build_lstm(units, input_shape, num_classes):
    """Build a stacked LSTM model with dropout and L2 regularisation."""
    model = Sequential()
    for i, u in enumerate(units):
        return_seq = (i < len(units) - 1)
        kwargs = dict(
            kernel_regularizer=l2(L2_REG),
            return_sequences=return_seq
        )
        if i == 0:
            kwargs['input_shape'] = input_shape
        model.add(LSTM(u, **kwargs))
        model.add(Dropout(DROPOUT))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    return model


# ─────────────────────────────────────────────
# Helper: compute per-class and macro FPR
# ─────────────────────────────────────────────
def compute_fpr(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    fpr_per_class = {}
    for i in range(n_classes):
        FP = cm[:, i].sum() - cm[i, i]
        TN = cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]
        fpr_per_class[i] = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    macro_fpr = np.mean(list(fpr_per_class.values()))
    return fpr_per_class, macro_fpr


# ─────────────────────────────────────────────
# Helper: get static model size in MB
# ─────────────────────────────────────────────
def get_model_size_mb(model):
    """Save model to a temp directory and return total size in MB."""
    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, "model.keras")
        model.save(path)
        total = 0
        if os.path.isdir(path):
            for dirpath, _, filenames in os.walk(path):
                for f in filenames:
                    total += os.path.getsize(os.path.join(dirpath, f))
        else:
            total = os.path.getsize(path)
    return total / (1024 ** 2)


# ─────────────────────────────────────────────
# Main sweep
# ─────────────────────────────────────────────
def run_sweep(dataset_path):
    print(f"\n{'='*65}")
    print(" SecureNet LSTM Stack Sweep — WSN-BFSF (4 classes)")
    print(f"{'='*65}")

    # ── Load data ──────────────────────────────────────────────
    print(f"\n[1/5] Loading dataset: {dataset_path}")
    df = pd.read_csv(dataset_path)
    print(f"      Shape: {df.shape}  |  Columns: {list(df.columns[-3:])}")

    # Identify class column (last column or 'Class')
    class_col = 'Class' if 'Class' in df.columns else df.columns[-1]
    X_raw = df.drop(columns=[class_col]).values
    y_raw = df[class_col].values

    # Encode labels
    le = LabelEncoder()
    y_enc = le.fit_transform(y_raw)
    print(f"      Classes: {dict(zip(le.classes_, range(len(le.classes_))))}")
    print(f"      Class distribution: {Counter(y_enc)}")

    # ── Outer 80/20 split (sealed test set) ────────────────────
    print("\n[2/5] Outer 80/20 train/test split (stratified, sealed test set)")
    X_outer_train, X_test_held, y_outer_train, y_test_held = train_test_split(
        X_raw, y_enc,
        test_size=OUTER_TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_enc
    )
    print(f"      Train: {X_outer_train.shape[0]}  |  Held-out test: {X_test_held.shape[0]}")

    # ── Preprocessing (impute + scale) on outer train ──────────
    imputer = SimpleImputer(strategy='mean')
    scaler  = StandardScaler()

    X_outer_train = imputer.fit_transform(X_outer_train)
    X_outer_train = scaler.fit_transform(X_outer_train)
    X_test_held   = imputer.transform(X_test_held)
    X_test_held   = scaler.transform(X_test_held)

    # ── SMOTE-NearMiss on full outer train (fold-contained in CV) ─
    # For the final model training (after CV selects best config),
    # we apply SMOTE-NearMiss on the full outer train.
    # Inside CV folds, SMOTE-NearMiss is applied per fold (see below).
    print("\n[3/5] RF feature selection on outer training set")
    # Quick RF on raw outer train to get top-k indices
    rf_selector = RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_selector.fit(X_outer_train, y_outer_train)
    importances = rf_selector.feature_importances_
    top_k_idx = np.argsort(importances)[::-1][:TOP_K]
    print(f"      Top-{TOP_K} feature indices: {top_k_idx}")

    X_outer_train_k = X_outer_train[:, top_k_idx]
    X_test_held_k   = X_test_held[:, top_k_idx]

    # Results container
    results = {}

    # ── Run each stack configuration ───────────────────────────
    print("\n[4/5] Training each stack configuration\n")
    skf = StratifiedKFold(
        n_splits=NUM_FOLDS, shuffle=True, random_state=RANDOM_STATE
    )

    for stack_name, units in STACKS.items():
        print(f"\n{'─'*55}")
        print(f"  Stack: {stack_name}  |  Units: {units}")
        print(f"{'─'*55}")

        fold_val_acc  = []
        fold_test_acc = []
        fold_recall_per_class = []
        fold_fpr_per_class    = []

        for fold, (tr_idx, val_idx) in enumerate(
            skf.split(X_outer_train_k, y_outer_train), 1
        ):
            # ── Fold split ──────────────────────────────────────
            Xf_tr = X_outer_train_k[tr_idx]
            Xf_val= X_outer_train_k[val_idx]
            yf_tr = y_outer_train[tr_idx]
            yf_val= y_outer_train[val_idx]

            # ── Fold-contained SMOTE-NearMiss ───────────────────
            try:
                sm = SMOTE(random_state=RANDOM_STATE)
                Xf_tr_sm, yf_tr_sm = sm.fit_resample(Xf_tr, yf_tr)
                nm = NearMiss()
                Xf_tr_res, yf_tr_res = nm.fit_resample(Xf_tr_sm, yf_tr_sm)
            except Exception as e:
                print(f"    Fold {fold}: SMOTE/NearMiss skipped ({e})")
                Xf_tr_res, yf_tr_res = Xf_tr, yf_tr

            # ── RF features → reshape for LSTM ──────────────────
            rf_fold = RandomForestClassifier(
                n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
            )
            rf_fold.fit(Xf_tr_res, yf_tr_res)

            Xf_tr_rf  = rf_fold.predict(Xf_tr_res).reshape(-1, 1, 1)
            Xf_val_rf = rf_fold.predict(Xf_val).reshape(-1, 1, 1)

            yf_tr_cat  = to_categorical(yf_tr_res,  NUM_CLASSES)
            yf_val_cat = to_categorical(yf_val,      NUM_CLASSES)

            # ── Build + train LSTM ───────────────────────────────
            tf.random.set_seed(RANDOM_STATE)
            model = build_lstm(units, (1, 1), NUM_CLASSES)

            history = model.fit(
                Xf_tr_rf, yf_tr_cat,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                validation_data=(Xf_val_rf, yf_val_cat),
                verbose=0
            )

            # ── Fold validation metrics ──────────────────────────
            val_acc = max(history.history['val_accuracy'])
            fold_val_acc.append(val_acc)

            # Fold predictions on validation set
            val_preds = np.argmax(model.predict(Xf_val_rf, verbose=0), axis=1)
            fold_test_acc.append(accuracy_score(yf_val, val_preds))
            fold_recall_per_class.append(
                recall_score(yf_val, val_preds,
                             labels=list(range(NUM_CLASSES)),
                             average=None,
                             zero_division=0)
            )
            fpr_cls, _ = compute_fpr(yf_val, val_preds, NUM_CLASSES)
            fold_fpr_per_class.append(
                [fpr_cls[i] for i in range(NUM_CLASSES)]
            )

            print(f"    Fold {fold:2d}: val_acc={val_acc*100:.4f}%  "
                  f"best_val={max(history.history['val_accuracy'])*100:.4f}%")

        # ── Aggregate across folds ───────────────────────────────
        mean_val_acc     = np.mean(fold_val_acc)
        mean_recall_cls  = np.mean(fold_recall_per_class, axis=0)
        worst_class_idx  = np.argmin(mean_recall_cls)
        worst_recall     = mean_recall_cls[worst_class_idx]
        mean_fpr_cls     = np.mean(fold_fpr_per_class, axis=0)
        macro_fpr        = np.mean(mean_fpr_cls)

        # ── Retrain final model on full outer train ──────────────
        print(f"\n  Retraining final {stack_name} model on full outer train...")
        try:
            sm_full = SMOTE(random_state=RANDOM_STATE)
            X_full_sm, y_full_sm = sm_full.fit_resample(
                X_outer_train_k, y_outer_train
            )
            nm_full = NearMiss()
            X_full_res, y_full_res = nm_full.fit_resample(X_full_sm, y_full_sm)
        except Exception as e:
            print(f"  SMOTE/NearMiss skipped ({e})")
            X_full_res, y_full_res = X_outer_train_k, y_outer_train

        rf_final = RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
        )
        rf_final.fit(X_full_res, y_full_res)

        X_full_rf    = rf_final.predict(X_full_res).reshape(-1, 1, 1)
        X_test_held_rf = rf_final.predict(X_test_held_k).reshape(-1, 1, 1)
        y_full_cat   = to_categorical(y_full_res, NUM_CLASSES)

        tf.random.set_seed(RANDOM_STATE)
        final_model = build_lstm(units, (1, 1), NUM_CLASSES)
        final_model.fit(
            X_full_rf, y_full_cat,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )

        # ── Held-out test evaluation ─────────────────────────────
        test_preds = np.argmax(
            final_model.predict(X_test_held_rf, verbose=0), axis=1
        )
        test_acc       = accuracy_score(y_test_held, test_preds) * 100
        test_recall_cls = recall_score(
            y_test_held, test_preds,
            labels=list(range(NUM_CLASSES)),
            average=None, zero_division=0
        )
        worst_test_idx  = np.argmin(test_recall_cls)
        worst_test_recall = test_recall_cls[worst_test_idx]
        test_fpr_cls, test_macro_fpr = compute_fpr(
            y_test_held, test_preds, NUM_CLASSES
        )
        test_fpr_pct   = test_macro_fpr * 100

        # ── Static model file size ───────────────────────────────
        model_mb = get_model_size_mb(final_model)

        # ── Store results ────────────────────────────────────────
        results[stack_name] = {
            "units":               units,
            "cv_mean_val_acc_pct": mean_val_acc * 100,
            "held_test_acc_pct":   test_acc,
            "cv_recall_per_class": {
                CLASS_NAMES[i]: float(mean_recall_cls[i]) * 100
                for i in range(NUM_CLASSES)
            },
            "test_recall_per_class": {
                CLASS_NAMES[i]: float(test_recall_cls[i]) * 100
                for i in range(NUM_CLASSES)
            },
            "worst_class_cv":   CLASS_NAMES[worst_class_idx],
            "worst_recall_cv_pct": worst_recall * 100,
            "worst_class_test": CLASS_NAMES[worst_test_idx],
            "worst_recall_test_pct": worst_test_recall * 100,
            "cv_fpr_per_class_pct": {
                CLASS_NAMES[i]: float(mean_fpr_cls[i]) * 100
                for i in range(NUM_CLASSES)
            },
            "test_fpr_per_class_pct": {
                CLASS_NAMES[i]: float(test_fpr_cls[i]) * 100
                for i in range(NUM_CLASSES)
            },
            "cv_macro_fpr_pct":   float(np.mean(mean_fpr_cls)) * 100,
            "test_macro_fpr_pct": test_fpr_pct,
            "model_size_mb":      model_mb,
            "within_2mb_budget":  model_mb < 2.0,
        }

        print(f"\n  ✓ {stack_name} — Held-out test results:")
        print(f"    ACC:          {test_acc:.4f}%")
        print(f"    Model size:   {model_mb:.4f} MB  "
              f"({'✓ within' if model_mb < 2.0 else '✗ exceeds'} <2 MB budget)")
        print(f"    Worst recall: {CLASS_NAMES[worst_test_idx]} "
              f"= {worst_test_recall*100:.4f}%")
        print(f"    Macro FPR:    {test_fpr_pct:.4f}%")

    # ── Final summary table ──────────────────────────────────────
    print(f"\n\n{'='*65}")
    print(" PARETO SWEEP SUMMARY — WSN-BFSF (held-out test set)")
    print(f"{'='*65}")
    hdr = f"{'Config':<12} {'ACC%':>8} {'Worst Recall%':>14} "  \
          f"{'Macro FPR%':>11} {'Size MB':>8} {'<2MB?':>6}"
    print(hdr)
    print('─' * 65)
    for name, r in results.items():
        marker = " ← Pareto-optimal" if name == "64/32/16" else ""
        print(
            f"{name:<12} "
            f"{r['held_test_acc_pct']:>8.4f} "
            f"{r['worst_recall_test_pct']:>14.4f} "
            f"{r['test_macro_fpr_pct']:>11.4f} "
            f"{r['model_size_mb']:>8.4f} "
            f"{'Yes' if r['within_2mb_budget'] else 'No':>6}"
            f"{marker}"
        )

    print(f"\n{'─'*65}")
    print(" Per-class recall on held-out test set:")
    print(f"{'─'*65}")
    print(f"{'Config':<12}", end="")
    for cn in CLASS_NAMES:
        print(f"  {cn:>12}", end="")
    print()
    print('─' * 65)
    for name, r in results.items():
        print(f"{name:<12}", end="")
        for cn in CLASS_NAMES:
            print(f"  {r['test_recall_per_class'][cn]:>11.4f}%", end="")
        print()

    print(f"\n{'─'*65}")
    print(" Per-class FPR on held-out test set:")
    print(f"{'─'*65}")
    print(f"{'Config':<12}", end="")
    for cn in CLASS_NAMES:
        print(f"  {cn:>12}", end="")
    print()
    print('─' * 65)
    for name, r in results.items():
        print(f"{name:<12}", end="")
        for cn in CLASS_NAMES:
            print(f"  {r['test_fpr_per_class_pct'][cn]:>11.4f}%", end="")
        print()

    # ── Save JSON results ────────────────────────────────────────
    out_path = "securenet_sweep_results.json"
    with open(out_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\n[5/5] Full results saved → {out_path}")
    print(f"{'='*65}\n")

    return results


# ─────────────────────────────────────────────
if __name__ == "__main__":
    # The argparse logic is meant for command-line execution.
    # In a Colab notebook, it's often more convenient to directly call the function.
    # We'll keep the argparse logic for flexibility but add a direct call.
    parser = argparse.ArgumentParser(
        description="SecureNet LSTM Stack Sweep (WSN-BFSF)"
    )
    parser.add_argument(
        "--dataset",
        # required=True, # No longer strictly required if we provide a default
        help="Path to WSN-BFSF CSV file (must contain a 'Class' column)"
    )

    # Check if a dataset path is provided as an argument
    if '--dataset' in sys.argv:
        args = parser.parse_args()
        dataset_path = args.dataset
    else:
        # Provide a default dataset path for Colab execution if no argument is given
        dataset_path = "/content/dataset.csv" # Assuming this file exists from your kernel state
        print(f"No --dataset argument provided. Using default path: {dataset_path}")

    if not os.path.exists(dataset_path):
        print(f"ERROR: Dataset not found: {dataset_path}. Please upload a dataset or specify the correct path.")
        sys.exit(1)

    run_sweep(dataset_path)
